# Project: Ocean Only Configuration - The Bare Minimum

This project is all about how we create the most basic ocean_only domain to shake things out, get spinning as fast as possible. Test everything before we get into our research question.

Download this project by either:

1. Running the CrocoDash CLI command: `crocodash template --machine derecho --notebook crocodash.projects.ocean_only`
2. Copying the file from your CrocoDash checkout: `demos/crocodash/projects/ocean_only`

Grid Generation - The first thing we are going to do is think about how to get a fast domain going, so I might look at some scaling data we've put online: https://crocodile-cesm.github.io/SeaSloth/mom6_scaling.html

Given how fast a 100x100 domain is, I think we should go with that, with maybe 40 cores for the Ocean. That'll run wicked fast.

We'll pick an arbitrary vertical grid.

In [ ]:
from CrocoDash.grid import Grid
from CrocoDash.vgrid import VGrid

grid = Grid(
  resolution = 0.05, # in degrees
  xstart = 278.0, # min longitude in [0, 360]
  lenx = 5.0, # longitude extent in degrees
  ystart = 7.0, # min latitude in [-90, 90]
  leny = 5.0, # latitude extent in degrees
  name = "basic_grid",
)


vgrid  = VGrid.hyperbolic(
    nk = 75, # number of vertical levels
    depth = 2000,
    ratio=20.0 # target ratio of top to bottom layer thicknesses
)

Now that we have our grids, we need to setup a bathymetry object. Normally, we'd use the `Topo.set_from_dataset` function to go from "obs" to a bathymetry on our grid. But we're trying to move with purpose, so we can use some Topo functions to create our own fun little domain.

In [ ]:
from CrocoDash.topo import Topo

topo = Topo(
    grid = grid,
    min_depth = 9.5, # in meters
)

topo.set_bowl(max_depth = 2000, dedge = 200)

topo.depth.plot()

Great! So I'm going to create my case now, and this step will hardly every change

In [ ]:
from pathlib import Path
from CrocoDash.case import Case

# CESM case (experiment) name
casename = "basic_ocean"

# CESM source root (Update this path accordingly!!!)
cesmroot ="<CESM>"

# Place where all your input files go 
inputdir = Path("<inputdir>") / casename
    
# CESM case directory
caseroot = Path("<casedir>") / casename


case = Case(
    cesmroot = cesmroot,
    caseroot = caseroot,
    inputdir = inputdir,
    ocn_grid = grid,
    ocn_vgrid = vgrid,
    ocn_topo = topo,
    project = "<PROJECT>", # Update this with your project code
    override = True,
    machine = "derecho",
    compset = "CR_JRA"
)

Now, the last step is to generate forcings. Which we do with a configure and process step. It always takes a long time to get raw data from other places, so we're going to use a fake data product. I also think we can run for 60 days. Doesn't sound too bad.

In [ ]:
case.configure_forcings(
    date_range = ["2019-01-01 00:00:00", "2019-03-03 00:00:00"],
    product_name="reference_ocean",
    function_name="get_reference_ocean_data"
)


In [ ]:
case.process_forcings()


Cool! Once this is done we have to `case.build` and `case.submit` in our case directory. Before that though, here's some things I'd `xmlchange` to run faster. 

1. I'd set the NTASKS=10, which sets every component to 10 cores, and makes the data atmosphere regridding very fast. (All components share these 10 cores)
2. I'd set the NTASKS_OCN=40 because of the scaling data we saw.
3. I'd set the ROOTPE=0, which ensures that all components share the same cores. If I set ROOTPE_OCN=10, it would skip the first 10 cores and run on the next 40 cores. 
4. I'd set the JOB_WALLCLOCK_TIME=00:30:00 --subgroup case.run, which ensures the queue picks up our job realllly fast. 


And there we go! A fake running ocean model! Isn't that awesome!